In [9]:
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import pandas as pd
import numpy as np
import shutil
import os

In [12]:
# Obtener la ruta donde kagglehub guarda los datos
path = kagglehub.dataset_download("henrysue/online-shoppers-intention")

print(f"La carpeta está en: {path}")

# Eliminar la carpeta completa para resetear el archivo dañado
try:
    shutil.rmtree(path)
    print("✅ Carpeta eliminada con éxito. Ya puedes volver a descargar el dataset.")
except Exception as e:
    print(f"❌ No se pudo eliminar: {e}")

La carpeta está en: C:\Users\setea\.cache\kagglehub\datasets\henrysue\online-shoppers-intention\versions\1
✅ Carpeta eliminada con éxito. Ya puedes volver a descargar el dataset.


In [13]:
path = kagglehub.dataset_download("henrysue/online-shoppers-intention")
file_path = os.path.join(path, "online_shoppers_intention.csv")

# 2. Leer con configuración "todoterreno"
df = pd.read_csv(
    file_path, 
    sep=None,             # Detecta automáticamente si es coma o punto y coma
    engine='python',      # El motor de Python es más flexible con errores de formato
    encoding='latin1',    # Evita el error de Unicode original
    on_bad_lines='skip'   # Si una línea está rota, la salta en lugar de detenerse
)

# 3. Verificar
print(f"Dataset cargado con {df.shape[0]} filas y {df.shape[1]} columnas.")
print(df.head())

100%|██████████| 252k/252k [00:00<00:00, 595kB/s]

Extracting files...
Dataset cargado con 12330 filas y 18 columnas.
   Administrative  Administrative_Duration  Informational  \
0               0                      0.0              0   
1               0                      0.0              0   
2               0                      0.0              0   
3               0                      0.0              0   
4               0                      0.0              0   

   Informational_Duration  ProductRelated  ProductRelated_Duration  \
0                     0.0               1                 0.000000   
1                     0.0               2                64.000000   
2                     0.0               1                 0.000000   
3                     0.0               2                 2.666667   
4                     0.0              10               627.500000   

   BounceRates  ExitRates  PageValues  SpecialDay Month  OperatingSystems  \
0         0.20       0.20         0.0         0.0   Feb             

In [ ]:
print("Valores nulos por columna:")
print(df.isnull().sum())

# 3. Eliminar duplicados
duplicados = df.duplicated().sum()
print(f"\nFilas duplicadas encontradas: {duplicados}")
df = df.drop_duplicates()

# 4. Ver los tipos de datos
print("\nTipos de datos:")
print(df.dtypes)

In [ ]:
print("Información del dataset:")
print(df.info())

print("Descripción de los datos:")
df.describe()

In [ ]:
# Configuración estética
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 5))

# Graficar la variable objetivo: ¿Cuántos compran (Revenue)?
plt.subplot(1, 3, 1)
sns.countplot(x='Revenue', data=df, palette='viridis')
plt.title('Distribución de Compras (Revenue)')

# Graficar tipos de visitantes
plt.subplot(1, 3, 2)
sns.countplot(x='VisitorType', data=df, palette='magma')
plt.title('Tipos de Visitantes')

# Graficar tráfico por mes
plt.subplot(1, 3, 3)
sns.countplot(x='Month', data=df, order=['Feb', 'Mar', 'May', 'June', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.title('Sesiones por Mes')

plt.tight_layout()
plt.show()

In [ ]:
print(df['Revenue'].value_counts())
print(df['VisitorType'].value_counts())
print(df['Month'].value_counts())

### 1. Revenue: 
- Hay un desequilibrio evidente, la mayoría de las sesiones terminan en no comprar el producto, 84.5% (10.422), mientras que las compras solo son 15.5% (1.908).
- Hay que hacer algo para ajustarlo.

### 2. Tipos de visitantes:
- Los visitantes que regresan (10.551) superan por mucho a los nuevos (1.694) y a los demás (85).
- La tienda depende fuertemente de la retención. Pero hay que tener un detalle en mente, que es que los que nuevos suelen tener una tasa de compra más alta por sesión (compra impulsiva).
- Como hay pocos de 'Other' podríamos ignorarlo.

### 3. Meses
- Mayo y Noviembre tienen mayor volumen de sesiones.
- Febrero tiene poco volumen, raro al tener San Valentín, es posible que haya mal marketing.

In [ ]:
# 1. Correlación entre variables numéricas
plt.figure(figsize=(12, 8))
# Solo columnas numéricas para la correlación
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, fmt=".2f", cmap='coolwarm')
plt.title('Matriz de Correlación de Variables')
plt.show()

### Correlaciones:
- Alta correlación 90%, entre 'BoundRates' y 'ExitRates', esto es normal ya que si un cliente sata, sale del sitio.
- Alta correlación 86%, entre 'ProductRelated' y 'ProductRelated_Duration', es decir, entre la cantidad de productos visitados y el tiempo que pasan en estos.
- La administración y su tiempo en esta también tienen alta correlación 60%.
- También tienen alta correlación las páginas informativas y su duración 62%.

In [ ]:
# Relación entre Fin de Semana y Compra
plt.figure(figsize=(10, 5))
sns.countplot(x='Weekend', data=df, palette='inferno')
plt.title('Compras: Fin de Semana vs Días de Diario')
plt.show()

In [ ]:
# Relación entre Fin de Semana y Compra
plt.figure(figsize=(10, 5))
sns.countplot(x='Weekend', hue='Revenue', data=df)
plt.title('Compras: Fin de Semana vs Días de Diario')
plt.show()

### Compras por finde semana y Diario.

- El volumen está en la semana: La gran mayoría de las sesiones ocurren entre lunes y viernes (Weekend: False). Esto sugiere que los usuarios navegan mientras trabajan o en sus rutinas diarias.

- No parece haber un "efecto fin de semana" que dispare la intención de compra; el comportamiento es bastante constante a lo largo de la semana.

In [ ]:
# Relación entre Fin de Semana y Compra
plt.figure(figsize=(10, 5))
sns.countplot(x='Month', hue='Revenue', data=df, order=['Feb', 'Mar', 'May', 'June', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'])
plt.title('Compras por meses')
plt.show()

In [ ]:
print(df['OperatingSystems'].value_counts())

In [ ]:
plt.rcParams['figure.figsize'] = (18, 7)
size = [6601, 2585, 2555, 589]
colors = ['red', 'blue', 'green', 'yellow']
labels = "2", "1", "3", "otros"

plt.subplot(1, 2, 2)
plt.pie(size, colors = colors, labels = labels, shadow = True, autopct = '%.2f%%', startangle=90)
plt.title('Distintos tipos de sistemas operativos', fontsize = 30)
plt.axis('off')
plt.legend()
plt.show()

In [ ]:
print(df['Browser'].value_counts())

In [ ]:
plt.rcParams['figure.figsize'] = (18, 7)
size = [7961, 2462, 736, 467, 704]
colors = ['red', 'blue', 'green', 'yellow', 'violet']
labels = "2", "1", "4", "5", "otros"

plt.subplot(1, 2, 2)
plt.pie(size, colors = colors, labels = labels, shadow = True, autopct = '%.2f%%', startangle=90)
plt.title('Distintos tipos de sistemas operativos', fontsize = 30)
plt.axis('off')
plt.legend()
plt.show()

In [ ]:
print(df['Region'].value_counts())

In [ ]:
plt.rcParams['figure.figsize'] = (18, 7)
size = [4780, 2403, 1182, 1136, 805, 761, 511, 434, 318]
colors = ['red', 'blue', 'green', 'yellow', 'violet', "magenta", "orange", "lightgreen", "pink"]
labels = "1", "3", "4", "2", "6", "7", "9", "8", "5"

plt.subplot(1, 2, 2)
plt.pie(size, colors = colors, labels = labels, shadow = True, autopct = '%.2f%%', startangle=90)
plt.title('Regiones', fontsize = 30)
plt.axis('off')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x='Revenue', y='PageValues', data=df)
plt.title('Influencia del Page Value en la Compra')
plt.show()

plt.figure(figsize=(8, 5))
sns.boxplot(x='Revenue', y='PageValues', data=df)
plt.title('Influencia del Page Value en la Compra')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x='ExitRates', y='ProductRelated_Duration', hue='Revenue', data=df, alpha=0.6)
plt.title('Duración en Productos vs Tasa de Salida')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='VisitorType', hue='Revenue', data=df)
plt.title('Compras vs tipos de visitantes')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(x='Month', y='SpecialDay', hue='Revenue', data=df)
plt.title('Meses vs Día especial')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.violinplot(x='Revenue', y='BounceRates', data=df, palette='magma')
plt.title('Distribución de Tasas de Rebote según Compra')
plt.show()

In [ ]:
orden_freq = df['TrafficType'].value_counts()
print(orden_freq)

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='TrafficType', hue='VisitorType', data=df, order=orden_freq.index)
plt.title('Origen del Tráfico según el Tipo de Visitante')
plt.legend(loc='upper right')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
# Usamos un scatter plot para ver la relación entre tiempo informativo y valor de página
sns.scatterplot(x='Informational_Duration', y='PageValues', hue='Revenue', data=df, alpha=0.6)
plt.title('Relación: Tiempo Informativo vs Valor de Página')
plt.show()